# 3.1 按时间切分

要预测的是"每个 SKU 下个月要多少"，所以先把订单摊成一张 SKU × 月的表：
一行 = 一个 SKU 的一个月，没有订单的月份需求量记 0（没卖出去也是信息，不能当成没这回事）。
再按月份先后切成训练、验证、最终评估三段——只能拿过去预测未来，不能打乱了随机切。

In [1]:
import json
import sys
from pathlib import Path

import pandas as pd

import dsflow

ROOT = Path.cwd()
while not (ROOT / "dsflow.yaml").is_file():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))
from demo_lib import FIRST_MONTH, LAST_MONTH, SPLITS, out, split_of

OUT = out("3.1")
CLEAN = out("1.2") / "orders_clean.parquet"
run = dsflow.start_run(
    "3.1", project=ROOT,
    hypothesis="按时间切分：训练 2024-07～2025-12、验证 2026-01～03、最终评估 2026-04～06，最终评估期不参与任何拟合与选择")

run.log_input(CLEAN, name="orders_clean")
clean = pd.read_parquet(CLEAN)
demand = clean[~clean["是否退货"]]
months = pd.period_range(FIRST_MONTH, LAST_MONTH, freq="M").strftime("%Y-%m")
skus = clean[["SKU", "类目"]].drop_duplicates("SKU")
print(f"{len(skus):,} 个 SKU × {len(months)} 个月 = {len(skus) * len(months):,} 行面板")


3,000 个 SKU × 24 个月 = 72,000 行面板


In [2]:
grid = skus.merge(pd.DataFrame({"月份": months}), how="cross")
agg = (demand.groupby(["SKU", "下单月份"], as_index=False)["数量"].sum()
       .rename(columns={"下单月份": "月份", "数量": "需求量"}))
panel = grid.merge(agg, on=["SKU", "月份"], how="left").fillna({"需求量": 0})
panel["需求量"] = panel["需求量"].astype("int64")
panel["划分"] = panel["月份"].map(split_of)
panel = panel.sort_values(["SKU", "月份"]).reset_index(drop=True)
run.log_output(panel, name="sku_month_panel", path=OUT / "panel.parquet", stage="splits",
               description="SKU×月需求量面板，一行 = 一个 SKU 的一个月，带划分标记")
panel.head(4)


,SKU,类目,月份,需求量,划分
0,SKU00001,办公通用物资,2024-07,0,训练
1,SKU00001,办公通用物资,2024-08,30,训练
2,SKU00001,办公通用物资,2024-09,23,训练
3,SKU00001,办公通用物资,2024-10,5,训练


In [3]:
summary = {name: {"月份": f"{a} 至 {b}", "行数": int((panel["划分"] == name).sum())} for name, (a, b) in SPLITS.items()}
summary["SKU数"] = int(len(skus))
summary["需求量为0的比例"] = round(float((panel["需求量"] == 0).mean()), 4)
(OUT / "split_summary.json").write_text(json.dumps(summary, ensure_ascii=False, indent=1), encoding="utf-8")
run.log_metrics({f"{k}行": v["行数"] for k, v in summary.items() if isinstance(v, dict)})
run.log_metrics({"SKU数": summary["SKU数"], "需求量为0的比例": summary["需求量为0的比例"]})
run.log_artifact(OUT / "split_summary.json", purpose="各划分的月份与行数", kind="table")
print(json.dumps(summary, ensure_ascii=False, indent=1))


{
 "训练": {
  "月份": "2024-07 至 2025-12",
  "行数": 54000
 },
 "验证": {
  "月份": "2026-01 至 2026-03",
  "行数": 9000
 },
 "最终评估": {
  "月份": "2026-04 至 2026-06",
  "行数": 9000
 },
 "SKU数": 3000,
 "需求量为0的比例": 0.2273
}


In [4]:
conclusion = (
    f"{summary['SKU数']:,} 个 SKU × {len(months)} 个月 = {len(panel):,} 行；"
    f"训练 {summary['训练']['行数']:,}、验证 {summary['验证']['行数']:,}、最终评估 {summary['最终评估']['行数']:,}；"
    f"需求量为 0 的 SKU 月占 {summary['需求量为0的比例']:.1%}"
)
run.set_conclusion(conclusion, validity="有效")
run.end()
print(conclusion)


3,000 个 SKU × 24 个月 = 72,000 行；训练 54,000、验证 9,000、最终评估 9,000；需求量为 0 的 SKU 月占 22.7%
